# Qwen3.5-4B — Baseline A1 (vLLM on Colab T4)

**Locked plan:** `docs/PLAN_QWEN_A_B_MIX.md`

| Setting | Value |
|---|---|
| Model | `Qwen/Qwen3.5-4B` (official — **not** GGUF/Unsloth) |
| Backend | **vLLM** |
| Dtype | **bfloat16** on T4 |
| Prompts | **200** antidoom-mix, **seed=42** |
| max_new_tokens | **4000** (not reduced) |
| temperature | **0.01** |
| Checkpoints | every prompt → local + **Google Drive** |
| Drive | `MyDrive/jlens_qwen_a1_baseline/` |

**This notebook is A1 only** (generate + loop detect). Exp1–3 come later.

**Resume:** if Colab dies, re-run Cells 1→6 then Cell 7; it restores from Drive and skips completed prompts.

**Time:** 4–5 h on T4 may not finish all 200 — that is OK; leave it running and resume later.

In [ ]:
# Cell 1 — GPU check + mount Drive
import os, subprocess
from pathlib import Path

assert Path("/content").exists(), "Run this on Google Colab"

gpu = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print("GPU:", gpu)
if "T4" not in gpu and "Tesla T4" not in gpu:
    print("WARNING: expected T4; continuing anyway. Prefer Runtime → T4.")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_OUT = Path("/content/drive/MyDrive/jlens_qwen_a1_baseline")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("DRIVE_OUT", DRIVE_OUT)
print("existing:", sorted(p.name for p in DRIVE_OUT.iterdir())[:20])

In [ ]:
# Cell 2 — clone / update repo (set BRANCH to the branch you pushed)
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Mithilyaganti/jlens-doom-loop-analysis.git"
BRANCH = "cursor/lfm2-exp1-jlens-fit"  # change if you push A1 to another branch
PROJECT = Path("/content/j-lens")

def run(cmd, **kw):
    print("+", " ".join(cmd))
    r = subprocess.run(cmd, text=True, capture_output=True, **kw)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(f"cmd failed: {cmd}")
    if r.stdout.strip():
        print(r.stdout.strip()[:2000])
    return r

if PROJECT.exists():
    print("Removing stale", PROJECT)
    shutil.rmtree(PROJECT, ignore_errors=True)

run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT)])
os.chdir(PROJECT)
rev = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("PROJECT", PROJECT, "git", rev, "branch", BRANCH)

# Sanity: A1 pieces present
assert (PROJECT / "jspace" / "drive_sync.py").is_file(), "drive_sync.py missing — wrong branch/commit"
assert (PROJECT / "results" / "prompt_sample_ids.json").is_file(), "prompt_sample_ids.json missing"
loading = (PROJECT / "jspace" / "loading.py").read_text(encoding="utf-8")
assert "JLENS_VLLM_TOKENIZER_ONLY" in loading, "tokenizer-only vLLM fix missing"
print("OK: drive_sync + prompt sample + vLLM tokenizer-only fix")

In [ ]:
# Cell 3 — install deps (vLLM + HF stack). Re-run after Runtime restart.
import os, sys, subprocess, shutil
from pathlib import Path

PROJECT = Path("/content/j-lens")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

%pip install -q -U "pandas>=2.1,<2.4" transformers accelerate datasets huggingface_hub \
    scipy tqdm pyyaml safetensors sentencepiece matplotlib

# vLLM — required for A1 on Colab
%pip install -q vllm

import vllm
print("vllm", getattr(vllm, "__version__", "?"))

# jlens on sys.path (outside git tree) — baseline A1 does not need it for vLLM gen,
# but vendor_bootstrap may import it.
JLENS_ROOT = Path("/content/open-jlens-data")
JLENS_PKG = JLENS_ROOT / "code" / "jacobian-lens"
MARKER = JLENS_PKG / "jlens" / "__init__.py"
if not MARKER.is_file():
    if JLENS_ROOT.exists():
        shutil.rmtree(JLENS_ROOT, ignore_errors=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/eliebak/open-jlens-data.git", str(JLENS_ROOT)],
        check=True,
    )
if str(JLENS_PKG) not in sys.path:
    sys.path.insert(0, str(JLENS_PKG))
print("jlens path OK", MARKER.is_file())

In [ ]:
# Cell 4 — environment (T4 A1 defaults). Do not lower max_new_tokens.
import os
from pathlib import Path

PROJECT = Path("/content/j-lens")
DRIVE_OUT = Path("/content/drive/MyDrive/jlens_qwen_a1_baseline")
os.chdir(PROJECT)

os.environ["PYTHONPATH"] = str(PROJECT)
os.environ["JLENS_MODEL"] = "Qwen/Qwen3.5-4B"
os.environ["JLENS_BACKEND"] = "vllm"
os.environ["JLENS_VLLM_DTYPE"] = "bfloat16"       # T4: start bf16
os.environ["JLENS_MAX_NEW_TOKENS"] = "4000"      # NOT nerfed
os.environ["JLENS_TEMPERATURE"] = "0.01"
os.environ["JLENS_BASELINE_PROMPTS"] = "200"
os.environ["JLENS_SAMPLE_SEED"] = "42"
os.environ["JLENS_TIER3_MAX"] = "0"              # A4 later
os.environ["JLENS_MAX_MODEL_LEN"] = "6000"
os.environ["JLENS_GPU_UTIL"] = "0.90"
os.environ["JLENS_DRIVE_SYNC_DIR"] = str(DRIVE_OUT)
os.environ["JLENS_DRIVE_SYNC_EVERY"] = "1"       # Drive sync after EVERY prompt
os.environ["JLENS_DRIVE_RESTORE"] = "1"
os.environ["JLENS_DRIVE_SKIP_HEAVY"] = "0"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("MODEL", os.environ["JLENS_MODEL"])
print("BACKEND", os.environ["JLENS_BACKEND"], "DTYPE", os.environ["JLENS_VLLM_DTYPE"])
print("MAX_NEW", os.environ["JLENS_MAX_NEW_TOKENS"], "SYNC_EVERY", os.environ["JLENS_DRIVE_SYNC_EVERY"])
print("DRIVE", os.environ["JLENS_DRIVE_SYNC_DIR"])
print("prompt sample", (PROJECT / "results" / "prompt_sample_ids.json").is_file())

In [ ]:
# Cell 5 — restore any previous Drive progress into /content/j-lens/results
import os, sys
from pathlib import Path

PROJECT = Path("/content/j-lens")
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

from jspace.drive_sync import restore_from_drive, drive_sync_dir
from jspace.model_config import get_active_model

cfg = get_active_model()
print("active", cfg.model_id, cfg.slug)
print("drive dir", drive_sync_dir())
ok = restore_from_drive(cfg.slug)
print("restored_from_drive", ok)

ckpt = PROJECT / "results" / "checkpoints" / f"baseline_pass_{cfg.slug}.json"
if ckpt.is_file():
    import json
    d = json.loads(ckpt.read_text())
    print("checkpoint:", {k: d.get(k) for k in ["n_completed", "n_loop", "n_total", "status", "saved_at"]})
else:
    print("no local checkpoint yet (fresh run)")

In [ ]:
# Cell 6 — SMOKE (1 prompt, still max_new=4000). Fix dtype here if this fails before Cell 7.
import os, sys, json
from pathlib import Path

PROJECT = Path("/content/j-lens")
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

os.environ["JLENS_BASELINE_SMOKE"] = "1"
os.environ["JLENS_BASELINE_SMOKE_N"] = "1"

!python -u scripts/02_baseline_pass.py

os.environ["JLENS_BASELINE_SMOKE"] = "0"

cfg_slug = "qwen3.5-4b"
ckpt = PROJECT / "results" / "checkpoints" / f"baseline_pass_{cfg_slug}.json"
print("ckpt exists", ckpt.is_file())
if ckpt.is_file():
    print(json.dumps(json.loads(ckpt.read_text()), indent=2)[:1200])
drive = Path(os.environ["JLENS_DRIVE_SYNC_DIR"])
print("Drive LAST_SYNC", (drive / "LAST_SYNC.txt").read_text() if (drive / "LAST_SYNC.txt").is_file() else "missing")
print("SMOKE DONE — if this looked healthy, run Cell 7")

In [ ]:
# Cell 7 — FULL 200-prompt baseline (resume-safe). Leave running.
import os, sys, json
from pathlib import Path

PROJECT = Path("/content/j-lens")
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

# Ensure smoke flag is off and settings are full-strength
os.environ["JLENS_BASELINE_SMOKE"] = "0"
os.environ["JLENS_MODEL"] = "Qwen/Qwen3.5-4B"
os.environ["JLENS_BACKEND"] = "vllm"
os.environ["JLENS_VLLM_DTYPE"] = os.environ.get("JLENS_VLLM_DTYPE", "bfloat16")
os.environ["JLENS_MAX_NEW_TOKENS"] = "4000"
os.environ["JLENS_TEMPERATURE"] = "0.01"
os.environ["JLENS_BASELINE_PROMPTS"] = "200"
os.environ["JLENS_SAMPLE_SEED"] = "42"
os.environ["JLENS_DRIVE_SYNC_EVERY"] = "1"
os.environ["JLENS_DRIVE_RESTORE"] = "1"

print("Starting full baseline…")
!python -u scripts/02_baseline_pass.py

summary = PROJECT / "results" / "baseline_pass_summary_qwen3.5-4b.json"
if summary.is_file():
    s = json.loads(summary.read_text())
    print("SUMMARY", {k: s.get(k) for k in ["n_completed", "n_loop", "loop_rate", "complete", "backend", "vllm_dtype", "max_new_tokens"]})
else:
    print("No summary yet — check logs / Drive checkpoint")

In [ ]:
# Cell 8 — status + force final Drive sync
import os, sys, json
from pathlib import Path

PROJECT = Path("/content/j-lens")
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

from jspace.drive_sync import sync_baseline_artifacts

slug = "qwen3.5-4b"
sync_baseline_artifacts(slug, reason="manual_cell8")

ckpt = PROJECT / "results" / "checkpoints" / f"baseline_pass_{slug}.json"
summary = PROJECT / "results" / f"baseline_pass_summary_{slug}.json"
log = PROJECT / "results" / f"trigger_gen_log_{slug}.jsonl"
drive = Path(os.environ.get("JLENS_DRIVE_SYNC_DIR", "/content/drive/MyDrive/jlens_qwen_a1_baseline"))

print("local ckpt", ckpt.is_file(), ckpt)
if ckpt.is_file():
    d = json.loads(ckpt.read_text())
    print({k: d.get(k) for k in ["n_completed", "n_loop", "n_total", "status", "loop_rate_so_far"]})
print("summary", summary.is_file())
print("log lines", sum(1 for _ in log.open()) if log.is_file() else 0)
print("Drive", drive)
print((drive / "LAST_SYNC.txt").read_text() if (drive / "LAST_SYNC.txt").is_file() else "no LAST_SYNC")
print("Drive results tree sample:", list((drive / "results").rglob("meta.json"))[:5] if (drive / "results").exists() else None)